In [ ]:
# ============================================================
# 🚀 SETUP — Ejecuta esta celda primero
# ============================================================
# Instala las dependencias necesarias para este bloque

import sys

# Detectar si estamos en Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -r https://raw.githubusercontent.com/shanglai/bsg_python_data_engineering/main/requirements-colab.txt -q
    print("✅ Dependencias instaladas")
else:
    print("ℹ️  Ejecutando localmente — asegúrate de tener el entorno activo")

print(f"Python {sys.version}")

## 📍 Contexto del bloque

**Capítulo 1:** Fundamentos de Python aplicado a datos
**Sección 2:** Pandas y mini ETL
**Bloque 1.2.3:** Transformaciones y generación de variables

**¿Qué vas a aprender?**
- Comprender el concepto central de este bloque en el pipeline de datos
- Aplicar las herramientas y técnicas del bloque con el dataset de transacciones
- Identificar errores comunes y cómo evitarlos
- Conectar lo aprendido con el bloque anterior y el siguiente

**¿Qué necesitas haber hecho antes?**
Haber ejecutado 1_2_2_Script.py

---
*BSG Institute · Python para Ingeniería de Datos · 2026*

==============================================================================
Capítulo 1: Fundamentos de Python aplicado a datos
Sección 2: Pandas y mini ETL
Bloque 3: Transformaciones y generación de variables
==============================================================================

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Configurar semilla para reproducibilidad
np.random.seed(987654)

In [ ]:
# ==============================================================================
# ST8, ST11: Contexto de negocio y preparación de datos iniciales
# Generar un dataset simulado de transacciones
# ==============================================================================
def generar_datos_transacciones():
    """
    Generar un DataFrame con datos de transacciones simuladas
    para aplicar transformaciones.
    """
    n_registros = 100
    
    # Generar fechas aleatorias en el último mes
    fechas = pd.date_range(start='2026-04-01', periods=n_registros, freq='6H')
    
    datos = {
        'id_transaccion': range(1001, 1001 + n_registros),
        'id_cliente': np.random.randint(10, 20, size=n_registros), # 10 clientes distintos
        'monto_bruto': np.round(np.random.uniform(50.0, 1500.0, size=n_registros), 2),
        'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar', 'Alimentos'], size=n_registros),
        'estado': np.random.choice(['Completado', 'Pendiente', 'Rechazado'], p=[0.7, 0.2, 0.1], size=n_registros),
        'fecha_transaccion': fechas
    }
    return pd.DataFrame(datos)

In [ ]:
df = generar_datos_transacciones()
print("--- Dataset Original ---")
print(df.head())
print("\n")

In [ ]:
# ==============================================================================
# ST1: Selección de columnas
# Seleccionar un subconjunto de columnas relevantes para el análisis
# ==============================================================================
print("--- ST1: Seleccionar columnas ---")
columnas_relevantes = ['id_transaccion', 'id_cliente', 'monto_bruto', 'estado', 'fecha_transaccion']
df_reducido = df[columnas_relevantes].copy()
print(df_reducido.head(3))
print("\n")

In [ ]:
# ==============================================================================
# ST2: Filtrado de registros
# Filtrar únicamente las transacciones que fueron completadas
# ==============================================================================
print("--- ST2: Filtrar registros ---")
df_completados = df[df['estado'] == 'Completado'].copy()
print(f"Total registros originales: {len(df)}")
print(f"Total registros completados: {len(df_completados)}")
print("\n")

In [ ]:
# ==============================================================================
# ST3: Operaciones vectorizadas
# ST4: Creación de columnas derivadas
# Calcular impuestos y monto neto usando operaciones vectorizadas (rápidas)
# ==============================================================================
print("--- ST3 y ST4: Operaciones vectorizadas y Columnas derivadas ---")
tasa_impuesto = 0.16

In [ ]:
# Calcular el impuesto multiplicando directamente la serie completa
df_completados['impuesto'] = df_completados['monto_bruto'] * tasa_impuesto

In [ ]:
# Crear la variable derivada del monto neto
df_completados['monto_neto'] = df_completados['monto_bruto'] - df_completados['impuesto']

In [ ]:
print(df_completados[['monto_bruto', 'impuesto', 'monto_neto']].head())
print("\n")

In [ ]:
# ==============================================================================
# ST5: Uso de apply
# Crear una lógica personalizada que no es fácilmente vectorizable
# ==============================================================================
print("--- ST5: Aplicar funciones custom con apply ---")

In [ ]:
def categorizar_monto(monto):
    """
    Clasificar el monto en categorías de ticket.
    """
    if monto < 200:
        return 'Ticket Bajo'
    elif monto <= 800:
        return 'Ticket Medio'
    else:
        return 'Ticket Alto'

In [ ]:
# Generar una nueva columna aplicando la función fila por fila
df_completados['tipo_ticket'] = df_completados['monto_neto'].apply(categorizar_monto)
print(df_completados[['monto_neto', 'tipo_ticket']].head())
print("\n")

In [ ]:
# ==============================================================================
# ST9: Feature engineering básico
# Extraer componentes de fechas para enriquecer los datos
# ==============================================================================
print("--- ST9: Realizar Feature Engineering básico ---")
# Extraer el día de la semana y la hora de la transacción
df_completados['dia_semana'] = df_completados['fecha_transaccion'].dt.day_name()
df_completados['hora'] = df_completados['fecha_transaccion'].dt.hour
print(df_completados[['fecha_transaccion', 'dia_semana', 'hora']].head())
print("\n")

In [ ]:
# ==============================================================================
# ST6: Cálculo de métricas
# ST7: Agrupaciones simples (groupby conceptual)
# ST8: Transformaciones basadas en lógica de negocio
# Generar un resumen por cliente para conocer su valor
# ==============================================================================
print("--- ST6, ST7 y ST8: Agrupar y Calcular métricas de negocio ---")

In [ ]:
# Agrupar por id_cliente y calcular la suma del monto neto y el conteo de transacciones
metricas_cliente = df_completados.groupby('id_cliente').agg(
    ingresos_totales=('monto_neto', 'sum'),
    numero_transacciones=('id_transaccion', 'count')
).reset_index()

In [ ]:
# Calcular el ticket promedio como métrica derivada del groupby
metricas_cliente['ticket_promedio'] = (
    metricas_cliente['ingresos_totales'] / metricas_cliente['numero_transacciones']
).round(2)

In [ ]:
# Ordenar los clientes por ingresos totales (lógica de negocio para ver los mejores)
metricas_cliente = metricas_cliente.sort_values(by='ingresos_totales', ascending=False)

In [ ]:
print(metricas_cliente.head())
print("\n")

In [ ]:
# ==============================================================================
# ST10: Validación de resultados transformados
# Hacer chequeos de cordura (sanity checks) para asegurar la calidad de datos
# ==============================================================================
print("--- ST10: Validar resultados transformados ---")

In [ ]:
# Verificar que no hay montos netos negativos
montos_invalidos = df_completados[df_completados['monto_neto'] < 0]
if montos_invalidos.empty:
    print("Validación exitosa: Todos los montos netos son mayores o iguales a cero.")
else:
    print("Alerta: Existen montos netos negativos.")

In [ ]:
# Verificar que el estado de los registros analizados sea 'Completado'
estados_unicos = df_completados['estado'].unique()
print(f"Estados presentes en el dataset final: {estados_unicos}")

In [ ]:
# Comprobar si la agrupación generó la misma cantidad de dinero que el total
ingresos_agrupados = metricas_cliente['ingresos_totales'].sum()
ingresos_totales = df_completados['monto_neto'].sum()

In [ ]:
# Usar math.isclose o round para evitar problemas de precisión de coma flotante
if round(ingresos_agrupados, 2) == round(ingresos_totales, 2):
    print(f"Validación exitosa: Total ingresos ({ingresos_totales:.2f}) coincide con agrupación.")
else:
    print("Error: Descuadre en las métricas agrupadas.")
print("\nFin del script.")